# Lightweight Walk-Forward Portfolio Optimizer

This notebook estimates robust weights for the eight selected holdings:

`SOXX, MRK, WMT, CVX, DUK, CME, LMT, PGR`

It deliberately avoids a large brute-force search. For each rolling window it:

1. trains on the previous five calendar years;
2. uses a small number of constrained SLSQP starting points;
3. tests the resulting weights on the following year without refitting;
4. aggregates the fold weights by their median;
5. produces a practical version rounded to 5-percentage-point increments; and
6. compares equal, robust, rounded, and full-sample-reference portfolios.

The robust recommendation is the **rounded walk-forward aggregate**, not the
full-sample optimum. Every export creates a new timestamped result folder.


## 1. Settings


In [ ]:
# If required, remove the leading # and run this once:
# %pip install pandas numpy scipy

from pathlib import Path
from datetime import datetime
import subprocess
import sys
import numpy as np
import pandas as pd

try:
    from scipy.optimize import minimize
except ModuleNotFoundError:
    print("SciPy is not installed in this Jupyter kernel; installing it now...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "scipy"])
    from scipy.optimize import minimize

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda value: f"{value:,.6f}")

# Store this notebook in the equity portfolio root folder before running it.
BASE_DIR = Path.cwd()
OUTPUT_DIR = BASE_DIR / "portfolio optimization results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ASSETS = ["SOXX", "MRK", "WMT", "CVX", "DUK", "CME", "LMT", "PGR"]
BENCHMARK = "SPY"

# Lightweight and editable optimization settings.
TRAIN_YEARS = 5
MIN_WEIGHT = 0.05
MAX_WEIGHT = 0.25
ROUNDING_STEP = 0.05
NUMBER_OF_STARTS = 10
RANDOM_SEED = 20260804
REBALANCE_FREQUENCY = "quarterly"

# Objective coefficients. All return/risk inputs are decimal values.
RETURN_REWARD = 1.00
VOLATILITY_PENALTY = 0.20
DRAWDOWN_PENALTY = 0.35
CORRELATION_PENALTY = 0.05
DOWNSIDE_EXCESS_REWARD = 0.25
EQUAL_WEIGHT_REGULARIZATION = 1.00

print("Portfolio root:", BASE_DIR.resolve())
print("Optimization history folder:", OUTPUT_DIR.resolve())


## 2. CSV paths


In [ ]:
CSV_PATHS = {
    "SOXX": BASE_DIR / "pharmaceutical" / "SOXX_10Y.csv",
    "MRK": BASE_DIR / "pharmaceutical" / "MRK_10Y.csv",
    "WMT": BASE_DIR / "consumer staples retail" / "WMT_10Y.csv",
    "CVX": BASE_DIR / "energy" / "CVX_10Y.csv",
    "DUK": BASE_DIR / "utilities" / "DUK_10Y.csv",
    "CME": BASE_DIR / "financial exchanges" / "CME_10Y.csv",
    "LMT": BASE_DIR / "aerospace defense" / "LMT_10Y.csv",
    "PGR": BASE_DIR / "property casualty insurance" / "PGR_10Y.csv",
    "SPY": BASE_DIR / "pharmaceutical" / "SPY_10Y.csv",
}

missing_files = [str(path) for path in CSV_PATHS.values() if not path.exists()]
if missing_files:
    raise FileNotFoundError("Missing required CSV files:\n" + "\n".join(missing_files))

display(pd.DataFrame({"Ticker": CSV_PATHS.keys(), "CSV path": map(str, CSV_PATHS.values())}))


## 3. Data, portfolio, and optimization functions


In [ ]:
def load_adjusted_close(path, ticker):
    data = pd.read_csv(path, encoding="utf-8-sig")
    normalized = {
        " ".join(str(column).replace("_", " ").strip().lower().split()): column
        for column in data.columns
    }
    required = {"date": "Date", "adj close": "Adj Close"}
    missing = [label for key, label in required.items() if key not in normalized]
    if missing:
        raise ValueError(f"{path.name}: missing columns {missing}; found {list(data.columns)}")

    data = data.rename(columns={normalized[key]: label for key, label in required.items()})
    data["Date"] = pd.to_datetime(data["Date"], errors="coerce", utc=True).dt.tz_localize(None)
    data["Adj Close"] = pd.to_numeric(data["Adj Close"], errors="coerce")
    if data["Date"].isna().any() or data["Date"].duplicated().any():
        raise ValueError(f"{path.name}: invalid or duplicate dates detected.")
    if data["Adj Close"].isna().any() or (data["Adj Close"] <= 0).any():
        raise ValueError(f"{path.name}: invalid adjusted prices detected.")

    series = data.sort_values("Date").set_index("Date")["Adj Close"]
    series.name = ticker
    return series


def validate_weights(weights):
    weights = np.asarray(weights, dtype=float)
    return (
        len(weights) == len(ASSETS)
        and np.isfinite(weights).all()
        and abs(weights.sum() - 1.0) <= 1e-6
        and (weights >= MIN_WEIGHT - 1e-8).all()
        and (weights <= MAX_WEIGHT + 1e-8).all()
    )


def rebalance_flags(index, frequency="quarterly"):
    if frequency == "quarterly":
        keys = index.to_period("Q")
    elif frequency == "monthly":
        keys = index.to_period("M")
    elif frequency == "annual":
        keys = index.to_period("Y")
    elif frequency == "none":
        return np.zeros(len(index), dtype=bool)
    else:
        raise ValueError("Unsupported rebalance frequency.")
    flags = np.zeros(len(index), dtype=bool)
    flags[1:] = np.asarray(keys[1:] != keys[:-1])
    return flags


def simulate_nav(prices, weights, initial_value=100.0, frequency="quarterly"):
    if not validate_weights(weights):
        raise ValueError("Weights violate the configured constraints.")
    prices = prices.dropna(how="any")
    returns = prices.pct_change(fill_method=None).fillna(0.0).to_numpy(dtype=float)
    flags = rebalance_flags(prices.index, frequency)
    holdings = initial_value * np.asarray(weights, dtype=float)
    nav = np.empty(len(prices), dtype=float)
    nav[0] = initial_value
    for position in range(1, len(prices)):
        holdings *= 1.0 + returns[position]
        nav[position] = holdings.sum()
        if flags[position]:
            holdings = nav[position] * np.asarray(weights, dtype=float)
    return pd.Series(nav, index=prices.index, name="Portfolio NAV")


def month_end_returns(nav):
    return nav.groupby(nav.index.to_period("M")).last().pct_change(fill_method=None).dropna()


def cagr(nav):
    elapsed_years = (nav.index[-1] - nav.index[0]).days / 365.2425
    if elapsed_years <= 0:
        return np.nan
    return float((nav.iloc[-1] / nav.iloc[0]) ** (1.0 / elapsed_years) - 1.0)


def maximum_drawdown(nav):
    return float((nav / nav.cummax() - 1.0).min())


In [ ]:
def evaluate_weights(prices, weights):
    asset_prices = prices[ASSETS]
    spy_prices = prices[BENCHMARK]
    nav = simulate_nav(asset_prices, weights, frequency=REBALANCE_FREQUENCY)
    spy_nav = spy_prices / spy_prices.iloc[0] * 100.0
    portfolio_monthly = month_end_returns(nav)
    spy_monthly = month_end_returns(spy_nav)
    pair = pd.concat(
        [portfolio_monthly.rename("Portfolio"), spy_monthly.rename("SPY")], axis=1
    ).dropna()
    negative_spy = pair["SPY"] < 0
    downside_excess = (
        pair.loc[negative_spy, "Portfolio"] - pair.loc[negative_spy, "SPY"]
    ).mean()
    if pd.isna(downside_excess):
        downside_excess = 0.0
    return {
        "CAGR": cagr(nav),
        "Annualized monthly volatility": float(portfolio_monthly.std(ddof=1) * np.sqrt(12)),
        "Maximum drawdown": maximum_drawdown(nav),
        "Correlation with SPY": float(pair["Portfolio"].corr(pair["SPY"])),
        "Average excess return in negative SPY months": float(downside_excess),
    }


def objective(weights, training_prices):
    if not validate_weights(weights):
        return 1e6
    metrics = evaluate_weights(training_prices, weights)
    equal_weight = 1.0 / len(ASSETS)
    concentration_penalty = float(np.sum((np.asarray(weights) - equal_weight) ** 2))
    score = (
        RETURN_REWARD * metrics["CAGR"]
        - VOLATILITY_PENALTY * metrics["Annualized monthly volatility"]
        - DRAWDOWN_PENALTY * abs(metrics["Maximum drawdown"])
        - CORRELATION_PENALTY * max(metrics["Correlation with SPY"], 0.0)
        + DOWNSIDE_EXCESS_REWARD * metrics["Average excess return in negative SPY months"]
        - EQUAL_WEIGHT_REGULARIZATION * concentration_penalty
    )
    return -float(score)


def feasible_random_weight(rng):
    # Start from equal weight and mix it with a Dirichlet draw until bounds hold.
    equal = np.full(len(ASSETS), 1.0 / len(ASSETS))
    for _ in range(10_000):
        candidate = 0.70 * equal + 0.30 * rng.dirichlet(np.ones(len(ASSETS)))
        if validate_weights(candidate):
            return candidate
    return equal.copy()


def optimize_training_window(training_prices, number_of_starts, rng):
    equal = np.full(len(ASSETS), 1.0 / len(ASSETS))
    starts = [equal] + [feasible_random_weight(rng) for _ in range(number_of_starts - 1)]
    bounds = [(MIN_WEIGHT, MAX_WEIGHT)] * len(ASSETS)
    constraints = [{"type": "eq", "fun": lambda weights: np.sum(weights) - 1.0}]
    candidates = []
    for start_number, start in enumerate(starts, start=1):
        result = minimize(
            objective,
            x0=start,
            args=(training_prices,),
            method="SLSQP",
            bounds=bounds,
            constraints=constraints,
            options={"maxiter": 250, "ftol": 1e-9, "disp": False},
        )
        if result.success and validate_weights(result.x):
            candidates.append((float(result.fun), result.x.copy(), start_number))
    if not candidates:
        raise RuntimeError("No optimization start converged for this training window.")
    candidates.sort(key=lambda item: item[0])
    best_objective, best_weights, best_start = candidates[0]
    return best_weights, -best_objective, best_start, len(candidates)


def project_to_constraints(target):
    target = np.asarray(target, dtype=float)
    equal = np.full(len(ASSETS), 1.0 / len(ASSETS))
    result = minimize(
        lambda weights: float(np.sum((weights - target) ** 2)),
        x0=equal,
        method="SLSQP",
        bounds=[(MIN_WEIGHT, MAX_WEIGHT)] * len(ASSETS),
        constraints=[{"type": "eq", "fun": lambda weights: np.sum(weights) - 1.0}],
        options={"ftol": 1e-12, "maxiter": 500},
    )
    if not result.success or not validate_weights(result.x):
        raise RuntimeError("Could not project aggregate weights into the feasible region.")
    return result.x


def round_weights_largest_remainder(weights, step=0.05):
    total_units = int(round(1.0 / step))
    min_units = int(np.ceil(MIN_WEIGHT / step - 1e-12))
    max_units = int(np.floor(MAX_WEIGHT / step + 1e-12))
    raw_units = np.asarray(weights) / step
    units = np.clip(np.floor(raw_units).astype(int), min_units, max_units)
    while units.sum() < total_units:
        eligible = np.where(units < max_units)[0]
        if len(eligible) == 0:
            raise RuntimeError("Rounding constraints cannot reach 100%.")
        choice = eligible[np.argmax(raw_units[eligible] - units[eligible])]
        units[choice] += 1
    while units.sum() > total_units:
        eligible = np.where(units > min_units)[0]
        if len(eligible) == 0:
            raise RuntimeError("Rounding constraints cannot reach 100%.")
        choice = eligible[np.argmax(units[eligible] - raw_units[eligible])]
        units[choice] -= 1
    rounded = units.astype(float) * step
    if not validate_weights(rounded):
        raise RuntimeError("Rounded weights violate constraints.")
    return rounded


## 4. Load and align the nine series


In [ ]:
all_prices = pd.concat(
    {ticker: load_adjusted_close(CSV_PATHS[ticker], ticker) for ticker in ASSETS + [BENCHMARK]},
    axis=1,
).sort_index()
aligned_prices = all_prices.dropna(how="any")
if len(aligned_prices) < 1_000:
    raise ValueError("Too few common observations after aligning all series.")

quality_summary = pd.DataFrame({
    "First aligned date": [aligned_prices.index.min()],
    "Last aligned date": [aligned_prices.index.max()],
    "Common trading days": [len(aligned_prices)],
    "Rows removed by alignment": [len(all_prices) - len(aligned_prices)],
})
display(quality_summary)


## 5. Five-year training / next-year validation


In [ ]:
rng = np.random.default_rng(RANDOM_SEED)
first_year = int(aligned_prices.index.min().year)
last_year = int(aligned_prices.index.max().year)
validation_years = range(first_year + TRAIN_YEARS, last_year + 1)

fold_rows = []
fold_weights = []
for validation_year in validation_years:
    training_start = pd.Timestamp(validation_year - TRAIN_YEARS, 1, 1)
    training_end = pd.Timestamp(validation_year - 1, 12, 31)
    validation_start = pd.Timestamp(validation_year, 1, 1)
    validation_end = pd.Timestamp(validation_year, 12, 31)
    training_prices = aligned_prices.loc[training_start:training_end]
    validation_prices = aligned_prices.loc[validation_start:validation_end]

    # Require roughly four trading years in train and three months in validation.
    if len(training_prices) < 1_000 or len(validation_prices) < 60:
        continue

    best_weights, training_score, best_start, converged_starts = optimize_training_window(
        training_prices, NUMBER_OF_STARTS, rng
    )
    training_metrics = evaluate_weights(training_prices, best_weights)
    validation_metrics = evaluate_weights(validation_prices, best_weights)
    fold_weights.append(best_weights)
    row = {
        "Validation year": validation_year,
        "Training start": training_prices.index.min(),
        "Training end": training_prices.index.max(),
        "Validation start": validation_prices.index.min(),
        "Validation end": validation_prices.index.max(),
        "Training objective score": training_score,
        "Winning start number": best_start,
        "Converged starts": converged_starts,
    }
    row.update({f"Train {key}": value for key, value in training_metrics.items()})
    row.update({f"Validation {key}": value for key, value in validation_metrics.items()})
    row.update({f"Weight {asset}": weight for asset, weight in zip(ASSETS, best_weights)})
    fold_rows.append(row)
    print(f"Completed fold ending in validation year {validation_year}.")

if len(fold_weights) < 2:
    raise RuntimeError("At least two valid walk-forward folds are required.")

fold_results = pd.DataFrame(fold_rows)
fold_weight_table = fold_results.set_index("Validation year")[[f"Weight {a}" for a in ASSETS]]
display(fold_results)
display(fold_weight_table.map(lambda value: f"{value:.2%}"))


## 6. Aggregate and round the robust weights


In [ ]:
fold_weight_array = np.vstack(fold_weights)
median_weights = np.median(fold_weight_array, axis=0)
robust_weights = project_to_constraints(median_weights)
rounded_weights = round_weights_largest_remainder(robust_weights, ROUNDING_STEP)
equal_weights = np.full(len(ASSETS), 1.0 / len(ASSETS))

# Full-sample optimization is included only as an overfitting/reference comparison.
full_sample_weights, full_sample_score, _, _ = optimize_training_window(
    aligned_prices, NUMBER_OF_STARTS, rng
)

recommended_weights = pd.DataFrame({
    "Equal weight benchmark": equal_weights,
    "Walk-forward median aggregate": robust_weights,
    "Recommended rounded weights": rounded_weights,
    "Full-sample optimum (reference only)": full_sample_weights,
}, index=ASSETS)
recommended_weights.index.name = "Ticker"
display(recommended_weights.map(lambda value: f"{value:.2%}"))
print("Recommended coefficients to copy into the validation notebook:")
print({asset: int(value) for asset, value in zip(ASSETS, (rounded_weights * 100).round())})


## 7. Compare candidate portfolios


In [ ]:
candidate_map = {
    "Equal weight benchmark": equal_weights,
    "Walk-forward median aggregate": robust_weights,
    "Recommended rounded weights": rounded_weights,
    "Full-sample optimum (reference only)": full_sample_weights,
}

full_sample_rows = []
for name, weights in candidate_map.items():
    row = {"Portfolio": name}
    row.update(evaluate_weights(aligned_prices, weights))
    full_sample_rows.append(row)
full_sample_comparison = pd.DataFrame(full_sample_rows).set_index("Portfolio")

# Fair comparison: evaluate each fixed candidate separately inside every validation year,
# then report the mean and worst validation-year result.
oos_rows = []
for fold in fold_rows:
    year = int(fold["Validation year"])
    validation_prices = aligned_prices.loc[f"{year}-01-01":f"{year}-12-31"]
    for name, weights in candidate_map.items():
        row = {"Portfolio": name, "Validation year": year}
        row.update(evaluate_weights(validation_prices, weights))
        oos_rows.append(row)
candidate_oos_by_year = pd.DataFrame(oos_rows)

candidate_oos_summary = candidate_oos_by_year.groupby("Portfolio").agg(
    Mean_validation_CAGR=("CAGR", "mean"),
    Worst_validation_CAGR=("CAGR", "min"),
    Mean_validation_volatility=("Annualized monthly volatility", "mean"),
    Worst_validation_drawdown=("Maximum drawdown", "min"),
    Mean_validation_SPY_correlation=("Correlation with SPY", "mean"),
    Mean_negative_SPY_month_excess=("Average excess return in negative SPY months", "mean"),
)

display(full_sample_comparison.map(lambda value: f"{value:.2%}"))
display(candidate_oos_summary.map(lambda value: f"{value:.2%}"))


## 8. Export a new timestamped result folder


In [ ]:
run_stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_DIR = OUTPUT_DIR / f"run_{run_stamp}"
suffix = 2
while RUN_DIR.exists():
    RUN_DIR = OUTPUT_DIR / f"run_{run_stamp}_{suffix:02d}"
    suffix += 1
RUN_DIR.mkdir(parents=False, exist_ok=False)

recommended_weights.to_csv(RUN_DIR / "recommended_portfolio_weights.csv")
fold_results.to_csv(RUN_DIR / "walk_forward_fold_results.csv", index=False)
candidate_oos_by_year.to_csv(RUN_DIR / "candidate_validation_results_by_year.csv", index=False)
candidate_oos_summary.to_csv(RUN_DIR / "candidate_validation_summary.csv")
full_sample_comparison.to_csv(RUN_DIR / "full_sample_candidate_comparison.csv")

settings = pd.Series({
    "Run timestamp": datetime.now().isoformat(timespec="seconds"),
    "Training years per fold": TRAIN_YEARS,
    "Minimum weight": MIN_WEIGHT,
    "Maximum weight": MAX_WEIGHT,
    "Rounding step": ROUNDING_STEP,
    "Number of SLSQP starts": NUMBER_OF_STARTS,
    "Random seed": RANDOM_SEED,
    "Rebalance frequency": REBALANCE_FREQUENCY,
    "Return reward": RETURN_REWARD,
    "Volatility penalty": VOLATILITY_PENALTY,
    "Drawdown penalty": DRAWDOWN_PENALTY,
    "Correlation penalty": CORRELATION_PENALTY,
    "Downside excess reward": DOWNSIDE_EXCESS_REWARD,
    "Equal-weight regularization": EQUAL_WEIGHT_REGULARIZATION,
    "Aligned first date": aligned_prices.index.min().date().isoformat(),
    "Aligned last date": aligned_prices.index.max().date().isoformat(),
}, name="Value")
settings.to_csv(RUN_DIR / "optimizer_settings.csv", header=True)

print("Generated optimization folder:", RUN_DIR.resolve())
for path in sorted(RUN_DIR.iterdir()):
    print(" -", path.name)


## Interpretation and safeguards

- **Recommended result:** use `Recommended rounded weights`. It is based on the
  median of weights learned in multiple earlier five-year windows and rounded to
  implementable 5% increments.
- **Full-sample optimum:** shown only to illustrate how an in-sample optimizer can
  differ from the walk-forward result. It should not be treated as the recommendation.
- **Regularization:** penalizes large deviations from 12.5% equal weight and reduces
  extreme concentration caused by noisy historical estimates.
- **Validation:** each fold's learned weights are evaluated on the following year.
  This preserves time order, but it is not a truly untouched test because the same
  ten-year dataset was previously used during stock selection.
- **Objective:** rewards CAGR and performance relative to SPY in negative SPY months;
  it penalizes volatility, absolute maximum drawdown, positive SPY correlation, and
  concentration. Coefficients are editable assumptions, not universal constants.
- **Costs:** taxes, spreads, slippage, and transaction costs are excluded.
